<a href="https://colab.research.google.com/github/nejumi/models-initial-course/blob/main/notebooks/01_basics/02_save_code_and_outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02. コードと出力ファイルを保存する

## この章のゴール
実行した Python コードと、そのコードが作った JSON / Markdown を同じ run に保存します。
Colab でもローカルでも動くよう、この notebook 内で小さなスクリプトを作ります。

`run.log_code()` はコード、`run.save()` は出力ファイルの保存に使います。


In [ ]:
# リポジトリ内ではルートを使い、Colab 単体では共通コードを取得します。
from pathlib import Path
import os
import sys
import urllib.request

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").is_file()), Path.cwd())
if not (root / "course_utils.py").exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/nejumi/models-initial-course/main/course_utils.py",
        root / "course_utils.py",
    )
os.chdir(root)
sys.path.insert(0, str(root))

from course_utils import setup_wandb
PROJECT, ENTITY, WANDB_INIT = setup_wandb()

## 保存するコードを作る
次のセルでスクリプトを作成します。その後、スクリプトを実行して出力を保存します。

In [ ]:
from pathlib import Path

out_dir = Path("outputs/02_save_code_and_outputs")
code_dir = out_dir / "source"
code_dir.mkdir(parents=True, exist_ok=True)
script_path = code_dir / "evaluate.py"
script_path.write_text(
    'from pathlib import Path\n'
    'import json\n'
    'out_dir = Path("outputs/02_save_code_and_outputs")\n'
    'out_dir.mkdir(parents=True, exist_ok=True)\n'
    'metrics = {"train_loss": 0.31, "val_acc": 0.88}\n'
    '(out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))\n'
    '(out_dir / "notes.md").write_text("# Demo output\\n\\nSaved with run.save.\\n")\n',
    encoding="utf-8",
)
print("保存対象のコード:", script_path)


## コードと出力ファイルを run に紐づける
このセルが、この章で覚える最小テンプレートです。

In [ ]:
import json
import subprocess
import sys
import wandb

cfg = {"model": "fixed-values-demo"}
with wandb.init(**WANDB_INIT, name="02-save-code-and-outputs", config=cfg, tags=["course", "save"]) as run:
    code_artifact = run.log_code(str(code_dir))
    assert code_artifact is not None, "保存対象のコードがありません"
    subprocess.run([sys.executable, str(script_path)], check=True)
    metrics = json.loads((out_dir / "metrics.json").read_text())
    run.log({"train/loss": metrics["train_loss"], "val/acc": metrics["val_acc"]})
    saved = run.save(str(out_dir / "*.json"), base_path=".", policy="end")
    saved += run.save(str(out_dir / "*.md"), base_path=".", policy="end")
    assert len(saved) == 2, "JSON と Markdown の保存対象を確認してください"
    run_url = run.url
print("Run URL:", run_url)


## W&B App で確認すること
- Files に metrics.json と notes.md がある
- Code に、実際に実行した evaluate.py がある
- API key や .env は保存対象に含めない
